# 1: Importação e Configuração

In [1]:
import pandas as pd
from pathlib import Path

# Configuração dos caminhos das pastas
PASTA_RAW = Path("../data/raw")

# Leitura dos dados brutos TUDO COMO TEXTO (dtype=str) para não mascarar erros
compras = pd.read_csv(PASTA_RAW / "compras.csv", dtype=str, keep_default_na=False)
itens = pd.read_csv(PASTA_RAW / "itens.csv", dtype=str, keep_default_na=False)
resultados = pd.read_csv(PASTA_RAW / "resultados.csv", dtype=str, keep_default_na=False)

print(f"compras.csv:    {len(compras)} linhas")
print(f"itens.csv:      {len(itens)} linhas")
print(f"resultados.csv: {len(resultados)} linhas")

compras.csv:    2042 linhas
itens.csv:      7077 linhas
resultados.csv: 15896 linhas


# 2: Auditoria de CNPJ/CPF (compras e resultados)

In [2]:
# Analisando tamanhos dos CNPJs dos órgãos (esperado 14 dígitos)
compras['tamanho_cnpj'] = compras['cnpj_orgao'].str.len()
print("Tamanhos de CNPJ encontrados em compras.csv:")
print(compras['tamanho_cnpj'].value_counts())

# Amostra dos CNPJs com tamanhos incorretos
compras[compras['tamanho_cnpj'] != 14][['cnpj_orgao', 'razao_social_orgao']].head(10)

Tamanhos de CNPJ encontrados em compras.csv:
tamanho_cnpj
14    1658
18     176
16     131
12      64
13      10
11       3
Name: count, dtype: int64


,cnpj_orgao,razao_social_orgao
3,119784000171,CONSELHO FEDERAL DE MEDICINA VETERINARIA
8,61.919.643/0002-09,CONSELHO FEDERAL DE ODONTOLOGIA
10,14.372.148/0001-61,COMPANHIA DOCAS DO ESTADO DA BAHIA
13,21.699.889/0001-17,CONSELHO REGIONAL DE ENFERMAGEM DE MINAS GERAIS
15,00.394.452/0001-03,COMANDO DO EXERCITO
24,91.335.315/0001-45,CONSELHO REGIONAL DE MEDICINA DO RIO GRANDE DO...
26,62.463.005/0001-08,CIA.DE ENTREPOSTOS E ARMAZENS GER.DE S.PAULO
50,10735145000194,"INSTITUTO FEDERAL DE EDUCACAO, CIENCIA E TECNO..."
53,13.738.204/0001-76,CONSELHO REGIONAL DE BIOMEDICINA - 5A. REGIAO
58,37115433000100,CONSELHO REGIONAL DE PSICOLOGIA DA 6 REGIAO - ...


# 3: Auditoria de Valores Monetários e Números (itens e resultados)

In [3]:
# Visualizando a formatação dos preços unitários estimados
print("Amostra de valores em texto:")
itens['valor_unitario_estimado'].sample(10, random_state=42)

Amostra de valores em texto:


5223       21.0
3053    1160.35
5329     2700.0
3698       13.9
3973        6.9
4534       55.0
408        18.3
1805     5131.0
2328    4362.31
6480     200.85
Name: valor_unitario_estimado, dtype: str

# 4: Auditoria de Nulos "Zumbis" (N/A, NULL, -)

In [4]:
# Verificando strings que representam nulos em colunas de texto
print("Valores mais comuns na coluna 'porte' de resultados.csv:")
print(resultados['porte'].value_counts().head(10))

Valores mais comuns na coluna 'porte' de resultados.csv:
porte
ME                7101
Não Informado     4415
Demais            3893
N/A                 72
NULL                64
n/a                 60
SEM INFORMAÇÃO      60
0                   57
NÃO INFORMADO       53
-                   52
Name: count, dtype: int64


# 5: Inconsistências de Cronograma (Homologação x Publicação)

In [5]:
# Cruzando datas de publicação e resultado para achar incoerências
df_datas = itens.merge(resultados, on=['numero_controle_pncp', 'numero_item'])
df_datas = df_datas.merge(compras, on='numero_controle_pncp')

# Filtrando casos onde a data de resultado vem antes da publicação
datas_invalidas = df_datas[df_datas['data_resultado'] < df_datas['data_publicacao']]
print(f"Itens com data de homologação anterior à publicação: {len(datas_invalidas)}")

Itens com data de homologação anterior à publicação: 4942
